In [1]:
from config import SEED, DATA

In [2]:
import numpy as np
import pandas as pd

from config import DATA
folder = DATA

X = pd.read_csv(folder + "/train/X_train.txt", sep=r"\s+", header=None).values
y = np.loadtxt(folder + "/train/y_train.txt", dtype=int)
groups = np.loadtxt(folder + "/train/subject_train.txt", dtype=int)

print(X.shape, y.shape, groups.shape)
print("people:", sorted(set(groups)))

(7352, 561) (7352,) (7352,)
people: [np.int64(1), np.int64(3), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(11), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(19), np.int64(21), np.int64(22), np.int64(23), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30)]


In [3]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=7)
# k could be 3 or 7 or 21 but balance sort of is in 7 only not undertraining not overtraining

for fold, (train_rows, val_rows) in enumerate(gkf.split(X, y, groups)):
    val_people = sorted(set(groups[val_rows]))
    train_people = set(groups[train_rows])
    shared = train_people & set(val_people)
    print("fold", fold,
          "| validation people:", [int(p) for p in val_people],
          "| validation windows:", len(val_rows),
          "| people on both sides:", shared)

fold 0 | validation people: [15, 22, 25] | validation windows: 1058 | people on both sides: set()
fold 1 | validation people: [3, 8, 21] | validation windows: 1030 | people on both sides: set()
fold 2 | validation people: [14, 26, 29] | validation windows: 1059 | people on both sides: set()
fold 3 | validation people: [1, 6, 30] | validation windows: 1055 | people on both sides: set()
fold 4 | validation people: [7, 19, 28] | validation windows: 1050 | people on both sides: set()
fold 5 | validation people: [5, 16, 27] | validation windows: 1044 | people on both sides: set()
fold 6 | validation people: [11, 17, 23] | validation windows: 1056 | people on both sides: set()


In [4]:
import os

fold_of_person = {}
for fold, (train_rows, val_rows) in enumerate(gkf.split(X, y, groups)):
    for p in set(groups[val_rows]):
        fold_of_person[int(p)] = fold

folds_table = pd.Series(fold_of_person, name="fold").sort_index()
folds_table.index.name = "person"

os.makedirs("results", exist_ok=True)
folds_table.to_csv("results/folds.csv")
print(folds_table)

person
1     3
3     1
5     5
6     3
7     4
8     1
11    6
14    2
15    0
16    5
17    6
19    4
21    1
22    0
23    6
25    0
26    2
27    5
28    4
29    2
30    3
Name: fold, dtype: int64


In [5]:
from sklearn.model_selection import cross_val_score
from sklearn.dummy import DummyClassifier

majority = DummyClassifier(strategy="most_frequent")

acc = cross_val_score(majority, X, y, groups=groups, cv=gkf, scoring="accuracy")
f1 = cross_val_score(majority, X, y, groups=groups, cv=gkf, scoring="f1_macro")

print("accuracy per fold:", acc.round(3), " mean:", acc.mean().round(3))
print("macro-F1 per fold:", f1.round(3), " mean:", f1.mean().round(3))

accuracy per fold: [0.205 0.2   0.185 0.168 0.205 0.188 0.189]  mean: 0.191
macro-F1 per fold: [0.057 0.056 0.052 0.048 0.057 0.053 0.053]  mean: 0.054


In [6]:
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from config import SEED

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))

random_split = KFold(n_splits=7, shuffle=True, random_state=SEED)

f1_random = cross_val_score(model, X, y, cv=random_split, scoring="f1_macro")
f1_person = cross_val_score(model, X, y, groups=groups, cv=gkf, scoring="f1_macro")

print("random window split: mean", f1_random.mean().round(3), " spread (std)", f1_random.std(ddof=1).round(3))
print("person split:        mean", f1_person.mean().round(3), " spread (std)", f1_person.std(ddof=1).round(3))
print("person split, per fold:", f1_person.round(3))

random window split: mean 0.985  spread (std) 0.007
person split:        mean 0.933  spread (std) 0.036
person split, per fold: [0.932 0.963 0.885 0.958 0.907 0.905 0.981]


## Cell 6: separating the two kinds of leakage, and a physics baseline

**Three splits, same model** (standardisation + logistic regression, C = 1):
1. **Random window split**: overlapping neighbours *and* the same people on both sides.
2. **Random split, neighbours removed**: the same random folds, but any training window that overlaps a validation window (its neighbour from the same person and activity) is dropped from training. The same people are still on both sides, but no readings are shared.
3. **Person split** (our saved folds): no shared readings and no shared people.

(1) − (2) is the part of the inflation caused by **overlap**; (2) − (3) is the part caused by **seeing the same people**.

**Physics baseline:** the same logistic regression on only **4 numbers per window** from Part 1: the mean of total acceleration on x, y and z (the direction of gravity, so posture) and the standard deviation of the body-acceleration size (how much the person moves). Scored on the saved person folds. It shows what the 561 engineered features add over the physics.

Every result is saved to `results/extra_experiments.csv` (one row per experiment and fold) and the website reads it from there.

In [7]:
from sklearn.model_selection import KFold, PredefinedSplit, cross_validate

saved_folds = PredefinedSplit(pd.read_csv("results/folds.csv", index_col="person")["fold"].loc[groups].values)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))

# windows are stored in time order, so row i overlaps row i-1 when both come from the same person and activity
same_bout_as_previous = np.r_[False, (groups[1:] == groups[:-1]) & (y[1:] == y[:-1])]

def random_split_neighbours_removed(n, k=7, seed=SEED):
    for train_rows, val_rows in KFold(n_splits=k, shuffle=True, random_state=seed).split(np.arange(n)):
        is_val = np.zeros(n, bool)
        is_val[val_rows] = True
        touches_val = np.zeros(n, bool)
        touches_val[1:] |= is_val[:-1] & same_bout_as_previous[1:]    # previous window is in validation
        touches_val[:-1] |= is_val[1:] & same_bout_as_previous[1:]    # next window is in validation
        yield train_rows[~touches_val[train_rows]], val_rows

purged = list(random_split_neighbours_removed(len(y)))
print("training windows kept after removing neighbours: %.0f%% on average"
      % (100 * np.mean([len(tr) for tr, _ in purged]) / (len(y) * 6 / 7)))

# physics features from the raw signals (Part 1): gravity direction and amount of motion
raw = {c: np.loadtxt(f"{DATA}/train/Inertial Signals/{c}_train.txt")
       for c in ["total_acc_x", "total_acc_y", "total_acc_z", "body_acc_x", "body_acc_y", "body_acc_z"]}
body_size = np.sqrt(raw["body_acc_x"] ** 2 + raw["body_acc_y"] ** 2 + raw["body_acc_z"] ** 2)
X_physics = np.column_stack([raw["total_acc_x"].mean(1), raw["total_acc_y"].mean(1), raw["total_acc_z"].mean(1),
                             body_size.std(1)])

experiments = {
    "random_split":                (X, KFold(n_splits=7, shuffle=True, random_state=SEED)),
    "random_split_neighbours_removed": (X, purged),
    "person_split":                (X, saved_folds),
    "physics_baseline":            (X_physics, saved_folds),
}
rows = []
for name, (inputs, cv) in experiments.items():
    out = cross_validate(model, inputs, y, cv=cv, scoring=["f1_macro", "accuracy"], n_jobs=-1)
    for fold in range(7):
        rows.append({"experiment": name, "fold": fold,
                     "macro_f1": out["test_f1_macro"][fold], "accuracy": out["test_accuracy"][fold]})
extra = pd.DataFrame(rows)
extra.to_csv("results/extra_experiments.csv", index=False)

summary = extra.groupby("experiment", sort=False).agg(
    macro_f1=("macro_f1", "mean"), f1_std=("macro_f1", lambda v: v.std(ddof=1)),
    accuracy=("accuracy", "mean"))
summary["misclassified_per_1000"] = (1000 * (1 - summary["accuracy"])).round().astype(int)
print(summary.round(3))

training windows kept after removing neighbours: 74% on average
                                 macro_f1  f1_std  accuracy  \
experiment                                                    
random_split                        0.985   0.007     0.984   
random_split_neighbours_removed     0.977   0.006     0.976   
person_split                        0.933   0.036     0.933   
physics_baseline                    0.821   0.068     0.828   

                                 misclassified_per_1000  
experiment                                               
random_split                                         16  
random_split_neighbours_removed                      24  
person_split                                         67  
physics_baseline                                    172  


“Evaluated with a random window split, logistic regression scores macro-F1 0.985; evaluated on unseen people it scores 0.933 ± 0.036. The random split hides most of the drop on new people and almost all of the person-to-person variation.”

# Module 3 summary: evaluation design

**Data used:** the 21 training people only. The 9 test people stay untouched until the final test (Part 7).

## Why this matters
It checks: an appropriate split, tuning kept separate from final testing, and justified metrics.

## Cell by cell

**Cell 1: load the model inputs**
- `X`: (7352, 561), one row per window, 561 features (the matrix X, n × d).
- `y`: (7352,), one activity label per window (the target).
- `groups`: (7352,), the person ID of each window. Used only for splitting, never as a model input.

**Cell 2: split by person with GroupKFold (7 folds)**
- 21 people → 7 groups of 3. Each round: train on 18 people, validate on the other 3. Repeat 7 times.
- Result: 3 people per fold, 1,030–1,059 windows per fold, and **no person on both sides in any fold**.
- Why k = 7: 21 people split evenly into groups of 3; each model trains on 18 of 21 people (close to the final model); only 7 fits per setting keeps tuning affordable. Leave-one-person-out (k = 21) was considered but not run (Part 7 explains why).

**Cell 3: save the folds (`results/folds.csv`)**
- A 21-row table: person → fold. It records the split so both of us validate every model on exactly the same people (a fair comparison), and so the split can be reproduced.
- Reproducibility: one seed (`SEED = 42` in `config.py`) for everything random; package versions pinned in `requirements.txt`.

**Cell 4: majority-class baseline, and why accuracy misleads**
- A "model" that always predicts the most common activity (lying, 19.1% of the training windows).
- Accuracy ≈ **0.19**, but macro-F1 ≈ **0.05** (F1 = 0.32 for lying, 0 for the other five).
- **Macro-F1** averages the F1 of all six activities equally, so a model cannot look good by ignoring hard activities. It is our main metric. The baseline every real model must beat: macro-F1 0.05.

**Cell 5: the leakage experiment (same model, two splits)**
- Model: standardisation + logistic regression, in a pipeline so the scaling is learned only from each fold's training people.

| Split | Mean macro-F1 | Spread (std over 7 folds) |
|---|---|---|
| Random window split (people on both sides) | **0.985** | 0.007 |
| Person split (our folds) | **0.933** | 0.036 |

**Cell 6: separating the two leaks, and a physics baseline**

| Split / model | Mean macro-F1 | Spread | Accuracy |
|---|---|---|---|
| Random window split | 0.985 | 0.007 | 0.984 |
| Random, overlapping neighbours removed | 0.977 | 0.006 | 0.976 |
| Person split | 0.933 | 0.036 | 0.933 |
| Physics baseline: 4 numbers, person split | 0.821 | 0.069 | 0.828 |

- Of the 0.052 gap between the random and person splits, removing the overlap closes only **0.008 (15%)**; the remaining **0.044 (85%) comes from having seen the same people**. The person identity is the bigger leak, not the 50% window overlap.
- The 561 engineered features are worth **+0.112 macro-F1** over gravity direction and motion size alone (0.933 vs 0.821).

- Accuracy and the share of windows misclassified for each split are computed in Cell 6 below (macro-F1 is not an error rate, so we do not read errors off 1 − macro-F1).
- The random split also hides person-to-person variation: the spread between folds is about 5× smaller (sample standard deviation over the 7 folds).
- Per-fold person-split scores: 0.932, 0.963, **0.885** (people 14, 26, 29: worst), 0.958, 0.907, 0.905, **0.981** (people 11, 17, 23: best).

## Why the random split is wrong for our question
Neighbouring windows share half their readings (Module 1: 95.5% of pairs), and every person's windows share that person's style. A random split puts near-copies and the same people on both sides, so it measures "recognising people already seen", not our question, "does it work for a new person?"

## Five takeaways
1. **All evaluation is by person**: a person's windows are never on both sides.
2. **The same 7 saved folds are used for every model**, so comparisons are fair.
3. **Macro-F1 is the main metric**; accuracy can flatter a useless model.
4. **A random split inflates the score** (0.985 vs 0.933) **and hides the variation between people.**
5. **Results differ a lot between people** (fold scores 0.885–0.981), so per-person results will be reported later.
6. **Seeing the same people matters more than overlapping windows**: 85% of the random split's inflation remains even after every overlapping training window is removed.